In [42]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chat_models import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.schema import BaseOutputParser


class CommaOutputParser(BaseOutputParser):
    def parse(self, text):
        items = text.strip().split(",")
        stripped_items = list(map(str.strip, items))
        return "".join(stripped_items)
    
def load_memory(_):
    return memory.load_memory_variables({})["history"]

def invoke_chain(question):
    res = chain.invoke({"question": question})
    result = res.content
    if question.startswith("Illustrate the movie "):
        p = CommaOutputParser()
        result = p.parse(res.content)
    memory.save_context(
        {"input": question},
        {"output": result},
    )
    print(result)


llm = ChatOpenAI(temperature=0.1)

memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=120,
    return_messages=True,
)

examples = [
    {
        'movie': 'Top Gun',
        'answer': '🛩️, 👨‍✈️, 🔥',
    },
    {
        'movie': 'The Godfather',
        'answer': '👨‍👨‍👦, 🔫, 🍝',
    }
    ,
    {
        'movie': 'Black Swan',
        'answer': '🖤, 🦢, 🩰',
    },
    {
        'movie': 'The Suicide Squad',
        'answer': '🤡, 🎭, 💣',
    }
]

example_prompt = ChatPromptTemplate.from_messages([
    ('human', 'Illustrate the movie <{movie}>.'),
    ('ai', '{answer}')
]) 

# 예제를 형식화
example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Reply only with three emojis that can represent the movie given. \
         In this case, DO NOT reply more than 3 emojis. Remember what the movie title was given \
         and answer if human ask about them."),
        example_prompt,
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}")
    ]
)

chain = RunnablePassthrough.assign(history=load_memory) | prompt | llm

invoke_chain("Illustrate the movie <Jungle Cruise>.")
invoke_chain("Illustrate the movie <Black Widow>.")
invoke_chain("Illustrate the movie <Free Guy>.")

memory.load_memory_variables({})



🌴🚢🐍
⚫️🕷️👩‍🦰
🎮😎🆓


{'history': [HumanMessage(content='Illustrate the movie <Jungle Cruise>.'),
  AIMessage(content='🌴🚢🐍'),
  HumanMessage(content='Illustrate the movie <Black Widow>.'),
  AIMessage(content='⚫️🕷️👩\u200d🦰'),
  HumanMessage(content='Illustrate the movie <Free Guy>.'),
  AIMessage(content='🎮😎🆓')]}

In [44]:
invoke_chain("What was the movie that the emojis ⚫️🕷️👩‍🦰 represent?")

The movie is "Black Widow."
